# Cat vs Dog Image Classification — Final Capstone Project
End-to-end: data audit → cleaning → preprocessing → traditional ML → feature engineering → CNN → augmentation → transfer learning → evaluation → deployment.
All **50 questions** are answered in order (Phase 1–9). Run top-to-bottom in **Google Colab** (Runtime → Change runtime type → **GPU**).

**Before running:** the Drive folder `https://drive.google.com/drive/folders/1k6vwjHI4LgOHIlnehn5lDBPienBf3J_J` must be visible inside *your* My Drive
(open the link → *Organize → Add shortcut to Drive*, or *Make a copy*). Colab's `drive.mount` only exposes My Drive. Then set `DATASET_DIR` below (or leave `None` to auto-detect the folder containing `Cat/` and `Dog/`).

## 0. Setup, configuration and reusable modules

In [ ]:
%pip install -q scikit-image
import os, sys, time, json, random, shutil, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED); np.random.seed(SEED)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ---------- CONFIG ----------
DATASET_DIR = os.environ.get('CDC_DATASET')   # e.g. '/content/drive/MyDrive/<your folder>'. None = auto-detect
QUICK = os.environ.get('CDC_QUICK') == '1'    # tiny smoke-test mode (leave off for the real run)
EPOCHS_CNN, EPOCHS_AUG, EPOCHS_TL, EPOCHS_FT = (1, 1, 1, 1) if QUICK else (15, 25, 8, 8)
SVM_N      = 300 if QUICK else 8000     # SVM trains on a stratified subset (kernel SVM is O(n^2)); same subset for every SVM
RF_TREES   = 20 if QUICK else 200
PCA_MAX    = 40 if QUICK else 600
PRETRAINED = None if QUICK else 'imagenet'
# -----------------------------

BASE = Path('/content') if IN_COLAB else Path('.').resolve()
PROJECT_DIR = Path('/content/drive/MyDrive/Cat-Dog-Classification') if IN_COLAB else BASE/'Cat-Dog-Classification'
RAW, CLEAN, PROC = BASE/'data'/'raw', BASE/'data'/'cleaned', BASE/'data'/'processed'
MODELS, RESULTS, FIG = PROJECT_DIR/'models', PROJECT_DIR/'results', PROJECT_DIR/'figures'
for d in [PROJECT_DIR/'src', PROJECT_DIR/'app', MODELS, RESULTS, FIG, RAW, CLEAN, PROC]:
    d.mkdir(parents=True, exist_ok=True)

def show(name):
    plt.tight_layout(); plt.savefig(FIG/name, dpi=110, bbox_inches='tight'); plt.show()

def write_module(name, text):
    (PROJECT_DIR/'src'/name).write_text(text.strip('\n') + '\n')
print('Colab:', IN_COLAB, '| project dir:', PROJECT_DIR)

### Reusable source modules (written to `src/`)
Preprocessing, cleaning, feature engineering, training and prediction code lives in Python modules, as the project rules require.

In [ ]:
write_module('preprocessing.py', r'''
# Shared image preprocessing: used by training notebooks AND the deployed app,
# so training-time and inference-time preprocessing are identical.
import io
import os
import numpy as np
from PIL import Image, ImageOps
from concurrent.futures import ThreadPoolExecutor

IMG_SIZE = 128          # CNN / transfer-learning input size
ML_SIZE = 64            # reduced size for traditional ML (64x64x3 = 12288 features)
CLASS_NAMES = ["Cat", "Dog"]
LABEL_MAP = {"Cat": 0, "Dog": 1}


class InvalidImageError(Exception):
    pass


def load_image_array(src, size=IMG_SIZE):
    # Open -> fix EXIF rotation -> force RGB (handles gray/RGBA/palette) -> resize -> uint8 array
    try:
        if hasattr(src, "read"):
            src = io.BytesIO(src.read())
        with Image.open(src) as im:
            im = ImageOps.exif_transpose(im)
            im = im.convert("RGB")
            im = im.resize((size, size), Image.BILINEAR)
            return np.asarray(im, dtype=np.uint8)
    except Exception as e:
        raise InvalidImageError(f"Cannot read image: {type(e).__name__}: {e}")


def _safe_load(path, size):
    return load_image_array(path, size)


def load_dataset_array(paths, size=IMG_SIZE, workers=8):
    with ThreadPoolExecutor(workers) as ex:
        arrs = list(ex.map(lambda p: _safe_load(p, size), paths))
    return np.stack(arrs).astype(np.uint8)


def downsample_2x(X):
    # exact 2x2 average-pooling (e.g. 128 -> 64), done in chunks to save memory
    n, h, w, c = X.shape
    out = np.empty((n, h // 2, w // 2, c), dtype=np.uint8)
    for i in range(0, n, 2000):
        b = X[i:i + 2000].astype(np.float32)
        b = b.reshape(len(b), h // 2, 2, w // 2, 2, c).mean(axis=(2, 4))
        out[i:i + 2000] = np.round(b).astype(np.uint8)
    return out


def normalize(X):
    # pixel range 0-255 -> 0-1
    return X.astype(np.float32) / 255.0
''')

In [ ]:
write_module('data_cleaning.py', r'''
import os
import shutil
import hashlib
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

warnings.filterwarnings("ignore")


def inspect_image(args):
    # Returns one metadata record; never raises (errors are recorded, not fatal)
    path, label = args
    rec = dict(path=path, label=label, ext=os.path.splitext(path)[1].lower(),
               file_size=os.path.getsize(path), width=np.nan, height=np.nan,
               channels=np.nan, mode=None, format=None, valid=False, error="", md5=None)
    try:
        if rec["file_size"] == 0:
            raise ValueError("empty file (0 bytes)")
        with Image.open(path) as im:
            rec.update(width=im.width, height=im.height, mode=im.mode,
                       format=im.format, channels=len(im.getbands()))
            im.verify()
        with Image.open(path) as im:   # verify() does not decode pixels; load() does
            im.load()
        rec["valid"] = True
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"[:200]
    try:
        with open(path, "rb") as f:
            rec["md5"] = hashlib.md5(f.read()).hexdigest()
    except Exception:
        pass
    return rec


def scan_dataset(root, class_names=("Cat", "Dog"), workers=16):
    # Recursively scan root/<class>/** and build a metadata DataFrame
    items = []
    for cls in class_names:
        for dp, _, files in os.walk(os.path.join(root, cls)):
            for fn in files:
                items.append((os.path.join(dp, fn), cls))
    with ThreadPoolExecutor(workers) as ex:
        recs = list(ex.map(inspect_image, items))
    return pd.DataFrame(recs)


def quarantine_invalid(df, log_csv, remove=False):
    # Record every invalid file in a CSV log; optionally delete them from disk
    bad = df[~df["valid"]].copy()
    bad.to_csv(log_csv, index=False)
    if remove:
        for p in bad["path"]:
            try:
                os.remove(p)
            except OSError:
                pass
    return bad


def build_cleaned_dataset(df, out_dir, min_side=32):
    # Drop invalid, conflicting-label duplicates, exact duplicates and tiny images; copy the rest
    steps = {"initial files": len(df)}
    d = df[df["valid"]].copy()
    steps["after removing corrupted / unreadable"] = len(d)
    nl = d.groupby("md5")["label"].nunique()
    conflict = set(nl[nl > 1].index)
    d = d[~d["md5"].isin(conflict)]
    steps["after removing cross-class duplicates"] = len(d)
    d = d.drop_duplicates("md5", keep="first")
    steps["after removing exact duplicates"] = len(d)
    d = d[(d["width"] >= min_side) & (d["height"] >= min_side)]
    steps[f"after removing images with side < {min_side}px"] = len(d)
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    new_paths = []
    for i, (p, lab, ext) in enumerate(zip(d["path"], d["label"], d["ext"])):
        dst_dir = os.path.join(out_dir, lab)
        os.makedirs(dst_dir, exist_ok=True)
        dst = os.path.join(dst_dir, f"{lab.lower()}_{i:05d}{ext}")
        shutil.copy2(p, dst)
        new_paths.append(dst)
    d = d.copy()
    d["orig_path"] = d["path"]
    d["path"] = new_paths
    return d.reset_index(drop=True), pd.Series(steps, name="images")
''')

In [ ]:
write_module('feature_engineering.py', r'''
import numpy as np
from joblib import Parallel, delayed


def flatten(X_float):
    return X_float.reshape(len(X_float), -1)


def to_grayscale(X_uint8):
    # ITU-R 601 luma, returns float32 (N, H, W) in [0, 1]
    w = np.array([0.299, 0.587, 0.114], dtype=np.float32)
    return (X_uint8.astype(np.float32) @ w) / 255.0


def hog_features(X_gray, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), n_jobs=-1):
    from skimage.feature import hog
    feats = Parallel(n_jobs=n_jobs)(
        delayed(hog)(g, orientations=orientations, pixels_per_cell=pixels_per_cell,
                     cells_per_block=cells_per_block, block_norm="L2-Hys") for g in X_gray)
    return np.asarray(feats, dtype=np.float32)
''')

In [ ]:
write_module('train_ml.py', r'''
import time
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)


def compute_metrics(y, pred, score):
    # Positive class = Dog (1)
    return dict(Accuracy=accuracy_score(y, pred),
                Precision=precision_score(y, pred, zero_division=0),
                Recall=recall_score(y, pred, zero_division=0),
                F1=f1_score(y, pred, zero_division=0),
                ROC_AUC=roc_auc_score(y, score))


def stratified_subsample(y, n, seed=42):
    # indices of a class-balanced-by-proportion subset of size n
    y = np.asarray(y)
    if n >= len(y):
        return np.arange(len(y))
    rng = np.random.RandomState(seed)
    idx = []
    for c in np.unique(y):
        ci = np.where(y == c)[0]
        k = int(round(n * len(ci) / len(y)))
        idx.extend(rng.choice(ci, size=min(k, len(ci)), replace=False))
    idx = np.array(idx)
    rng.shuffle(idx)
    return idx


def sk_scores(model, X):
    pred = model.predict(X)
    if hasattr(model, "predict_proba"):
        score = model.predict_proba(X)[:, 1]
    elif hasattr(model, "decision_function"):
        score = model.decision_function(X)
    else:
        score = pred.astype(float)
    return pred, score


def timed_fit(model, X, y):
    t0 = time.time()
    model.fit(X, y)
    return model, time.time() - t0
''')

In [ ]:
write_module('train_cnn.py', r'''
import time
import numpy as np
import keras
from keras import layers

SEED = 42

# Three experimental CNN designs (Q46). "A" is the baseline architecture from the project brief.
ARCHS = {
    "CNN-A (3 conv, 3x3, drop 0.4)": dict(filters=(32, 64, 128), kernel=3, dense=128, dropout=0.4, bn=False, gap=False),
    "CNN-B (4 conv, BN, GAP, drop 0.5)": dict(filters=(32, 64, 128, 256), kernel=3, dense=128, dropout=0.5, bn=True, gap=True),
    "CNN-C (3 conv, 5x5, small, drop 0.3)": dict(filters=(16, 32, 64), kernel=5, dense=64, dropout=0.3, bn=False, gap=False),
}


def augmentation_layers():
    # Active only when training=True, so validation / test / inference are never augmented
    aug = [layers.RandomFlip("horizontal"),
           layers.RandomRotation(0.06),
           layers.RandomZoom(0.15),
           layers.RandomTranslation(0.1, 0.1)]
    try:
        aug.append(layers.RandomShear(x_factor=0.1, y_factor=0.1))
    except Exception:
        pass
    return keras.Sequential(aug, name="augmentation")


def build_cnn(cfg, input_shape=(128, 128, 3), augment=False, name="cnn"):
    inp = keras.Input(shape=input_shape)
    x = layers.Rescaling(1.0 / 255)(inp)           # pixel normalization 0-255 -> 0-1 lives inside the model
    if augment:
        x = augmentation_layers()(x)
    for f in cfg["filters"]:
        x = layers.Conv2D(f, cfg["kernel"], padding="same", use_bias=not cfg["bn"])(x)
        if cfg["bn"]:
            x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.GlobalAveragePooling2D()(x) if cfg["gap"] else layers.Flatten()(x)
    x = layers.Dense(cfg["dense"], activation="relu")(x)
    x = layers.Dropout(cfg["dropout"])(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inp, out, name=name)


def build_transfer_model(pretrained="imagenet", input_shape=(128, 128, 3), dropout=0.3):
    base = keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights=pretrained)
    base.trainable = False                         # frozen backbone for the first baseline
    inp = keras.Input(shape=input_shape)
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0)(inp)   # MobileNetV2 expects [-1, 1]
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inp, out, name="mobilenetv2_transfer"), base


def unfreeze_top_layers(base, n=30):
    base.trainable = True
    for l in base.layers[:-n]:
        l.trainable = False
    for l in base.layers:                           # keep BatchNorm statistics frozen
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False


def compile_and_fit(model, name, X_tr, y_tr, X_va, y_va, model_dir, epochs=20,
                    lr=1e-3, patience=4, batch_size=64):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
                  metrics=["accuracy", keras.metrics.AUC(name="auc")])
    safe = "".join(c if c.isalnum() else "_" for c in name)
    cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
           keras.callbacks.ModelCheckpoint(f"{model_dir}/{safe}.keras", monitor="val_loss", save_best_only=True),
           keras.callbacks.CSVLogger(f"{model_dir}/{safe}_log.csv")]
    t0 = time.time()
    hist = model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=epochs,
                     batch_size=batch_size, callbacks=cbs, shuffle=True, verbose=2)
    return hist, time.time() - t0
''')

In [ ]:
write_module('predict.py', r'''
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent))
import numpy as np
from preprocessing import load_image_array, InvalidImageError, IMG_SIZE, CLASS_NAMES


def load_trained_model(path):
    import keras
    return keras.models.load_model(str(path))


def predict_image(model, src, threshold=0.5):
    # Same preprocessing as training (shared function) -> sigmoid probability of "Dog"
    arr = load_image_array(src, IMG_SIZE)
    x = arr[None].astype("float32")            # normalization happens inside the model
    p_dog = float(model.predict(x, verbose=0).ravel()[0])
    label = CLASS_NAMES[1] if p_dog >= threshold else CLASS_NAMES[0]
    return dict(label=label, prob_dog=p_dog, prob_cat=1 - p_dog,
                confidence=max(p_dog, 1 - p_dog), image=arr)
''')

In [ ]:
sys.path.insert(0, str(PROJECT_DIR/'src'))
import importlib, preprocessing, data_cleaning, feature_engineering, train_ml, train_cnn, predict
from preprocessing import *
import tensorflow as tf, keras
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

# Phase 1 — Problem Understanding

**Q1. Objective.** Build a model that takes a photograph and predicts whether it contains a **Cat** or a **Dog**. Formally: learn a function f(image) → {Cat, Dog} from labelled examples, such that it generalises to unseen images. The solution is developed progressively (traditional ML → CNN → transfer learning) and finally deployed as an app.

**Q2. Type of learning.** **Supervised learning**: every training image comes with a ground-truth label (its folder name), and the model learns the mapping from input to label by minimising prediction error. It is not unsupervised (no labels would be used) and not reinforcement learning (no agent/reward/environment interaction).

**Q3. Binary vs multiclass.** **Binary classification** — exactly two mutually exclusive classes (Cat = 0, Dog = 1). A single sigmoid output unit giving P(Dog) is sufficient.

**Q4. Variables.** *Independent variable (X)*: the image pixels (H × W × 3 intensity values), or features derived from them (flattened pixels, HOG, PCA components). *Dependent variable (y)*: the class label (Cat / Dog → 0 / 1).

**Q5. Applications.** Pet-adoption / lost-and-found photo tagging, automatic photo-gallery organisation, pet-shelter intake, smart pet doors (let the cat in, not the dog), wildlife / camera-trap filtering, content moderation and search for pet marketplaces, veterinary-clinic record systems, and as a teaching template for any binary image classifier (e.g. defect / no defect).

# Phase 2 — Dataset Acquisition & Understanding

In [ ]:
# Q6: locate the dataset in Drive and examine its directory structure
def find_dataset_root(base):
    for root, dirs, _ in os.walk(base):
        low = {d.lower(): d for d in dirs}
        if 'cat' in low and 'dog' in low:
            return Path(root), low['cat'], low['dog']
    raise FileNotFoundError(f'No folder containing Cat/ and Dog/ found under {base}. Set DATASET_DIR.')

import zipfile
def find_zip(base):
    for root, _, files in os.walk(base):
        for f in files:
            if f.lower().endswith('.zip') and 'cat' in f.lower() and 'dog' in f.lower():
                return Path(root)/f
    return None

if DATASET_DIR:
    bases = [DATASET_DIR]
elif IN_COLAB:
    tops = [p for p in Path('/content/drive/MyDrive').iterdir() if p.is_dir() and 'cat' in p.name.lower() and 'dog' in p.name.lower()]
    bases = [str(p) for p in tops] + ['/content/drive/MyDrive']
else:
    bases = ['.']
SRC_ROOT = None
for b in bases:                      # 1) loose Cat/ Dog/ folders  2) a ZIP containing them (extracted to local disk)
    try:
        SRC_ROOT, CAT_DIR, DOG_DIR = find_dataset_root(b); break
    except FileNotFoundError:
        z = find_zip(b)
        if z is not None:
            EXTRACT = BASE/'data'/'zip_extract'
            print('Dataset is a ZIP:', z, '-> extracting to', EXTRACT, '(takes a few minutes)')
            with zipfile.ZipFile(z) as zf: zf.extractall(EXTRACT)
            SRC_ROOT, CAT_DIR, DOG_DIR = find_dataset_root(EXTRACT); break
if SRC_ROOT is None:
    raise FileNotFoundError('Could not find Cat/Dog folders or a cats-and-dogs ZIP in Drive. Set DATASET_DIR.')
print('Dataset root:', SRC_ROOT)
for entry in sorted(SRC_ROOT.iterdir()):
    n = sum(len(f) for _, _, f in os.walk(entry)) if entry.is_dir() else 1
    print(f"  {'[DIR]' if entry.is_dir() else '[file]'} {entry.name:<25} {n} files")
print('\nClasses present: Cat, Dog')

In [ ]:
# Copy to local disk once (reading thousands of small files from Drive repeatedly is very slow)
for cls, folder in [('Cat', CAT_DIR), ('Dog', DOG_DIR)]:
    if not (RAW/cls).exists():
        shutil.copytree(SRC_ROOT/folder, RAW/cls)
print({c: sum(len(f) for _, _, f in os.walk(RAW/c)) for c in ['Cat','Dog']})

In [ ]:
# Q7-Q9: initial counts (all files found) and balance
counts = {c: sum(len(f) for _, _, f in os.walk(RAW/c)) for c in ['Cat','Dog']}
total = sum(counts.values())
print(f"Q7  Cat images (initial scan): {counts['Cat']}")
print(f"Q8  Dog images (initial scan): {counts['Dog']}")
for c, n in counts.items():
    print(f"Q9  {c}: {n/total*100:.2f}%")
print('Q9  Balanced?', 'YES (near 50/50)' if abs(counts['Cat']-counts['Dog'])/total < 0.05 else 'NO - imbalanced')

In [ ]:
# Q10: recursive scan -> metadata DataFrame (path, label, width, height, channels, extension, ...)
t0 = time.time()
df = data_cleaning.scan_dataset(str(RAW))
print(f'Scanned {len(df)} files in {time.time()-t0:.0f}s')
df.to_csv(RESULTS/'metadata_raw.csv', index=False)
df[['path','label','width','height','channels','ext','file_size','mode','valid']].head(10)

In [ ]:
# Q11: unique image file formats
print('File extensions:\n', df['ext'].value_counts(), '\n')
print('PIL-detected formats:\n', df['format'].value_counts(dropna=False))

In [ ]:
# Q12: min / max / mean / median image dimensions (readable images only)
v = df[df.valid]
stats = pd.DataFrame({c: [v[c].min(), v[c].max(), v[c].mean(), v[c].median()] for c in ['width','height']},
                     index=['min','max','mean','median'])
stats['file_size_KB'] = [v.file_size.min()/1024, v.file_size.max()/1024, v.file_size.mean()/1024, v.file_size.median()/1024]
stats.round(1)

In [ ]:
# Q13: distribution of widths / heights
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(v['width'], bins=50, ax=ax[0], color='steelblue').set_title('Image width')
sns.histplot(v['height'], bins=50, ax=ax[1], color='darkorange').set_title('Image height')
ax[2].scatter(v['width'], v['height'], s=3, alpha=.3); ax[2].set_title('Width vs height'); ax[2].set_xlabel('width'); ax[2].set_ylabel('height')
show('q13_size_distribution.png')
top = v.groupby(['width','height']).size().sort_values(ascending=False).head(5)
print('Most common sizes (w,h):\n', top)
print(f"\nObservation: sizes are NOT uniform (widths {int(v.width.min())}-{int(v.width.max())}, heights {int(v.height.min())}-{int(v.height.max())}); "
      f"the distribution is right-skewed with a dominant cluster around {int(v.width.median())}x{int(v.height.median())} and a long tail of large images. "
      "Hence every image must be resized to one shape before modelling.")

In [ ]:
# Q14: 20 random images with labels
from PIL import Image
rng = np.random.RandomState(SEED)
samp = v.sample(20, random_state=SEED)
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
for a, (_, r) in zip(axes.ravel(), samp.iterrows()):
    with Image.open(r.path) as im: a.imshow(im.convert('RGB'))
    a.set_title(f"{r.label}  ({int(r.width)}x{int(r.height)})"); a.axis('off')
show('q14_random_20.png')

In [ ]:
# Q15: Cat (left 3 columns) vs Dog (right 3 columns) side by side
cats = v[v.label=='Cat'].sample(12, random_state=1).path.tolist()
dogs = v[v.label=='Dog'].sample(12, random_state=1).path.tolist()
fig, axes = plt.subplots(4, 6, figsize=(16, 11))
for r in range(4):
    for c in range(3):
        for off, lst, name in [(0, cats, 'Cat'), (3, dogs, 'Dog')]:
            a = axes[r, c+off]
            with Image.open(lst[r*3+c]) as im: a.imshow(im.convert('RGB'))
            a.set_title(name); a.axis('off')
show('q15_cat_vs_dog_grid.png')

**Q15 – Visual differences observed:** cats typically have smaller, rounder heads, short pointed triangular ears, vertical slit pupils, prominent whiskers and a compact, often curled body. Dogs show far more variety in size and breed — longer muzzles/snouts, floppy or varied ears, wider range of fur patterns and body shapes. Backgrounds (indoor sofas for cats, outdoor grass for dogs) also correlate with class, which is a potential **shortcut/bias** a model might exploit. High within-class variability and some cross-class similarity (e.g. small fluffy dogs) is what makes the task non-trivial for raw-pixel methods.

# Phase 3 — Data Quality & Image Cleaning

In [ ]:
# Q16: corrupted / unreadable / invalid files
bad = df[~df.valid]
print(f'Q16  Problematic files found: {len(bad)}  ({len(bad)/len(df)*100:.2f}% of all files)')
print(bad.groupby('label').size().rename('count').to_frame(), '\n')
print(bad['error'].str.split(':').str[0].value_counts().rename('error type').to_frame())
bad[['path','label','file_size','error']].head(10)

In [ ]:
# Q17: function that detects and records (or removes) corrupted images -> data_cleaning.inspect_image / quarantine_invalid
import inspect
print(inspect.getsource(data_cleaning.inspect_image)[:900], '...\n')
bad_log = data_cleaning.quarantine_invalid(df, str(RESULTS/'corrupted_images_log.csv'), remove=False)
print(f'Recorded {len(bad_log)} invalid files to results/corrupted_images_log.csv (originals left untouched; they are excluded from the cleaned copy).')

In [ ]:
# Q18: duplicate detection (exact byte-level duplicates via MD5)
vv = df[df.valid]
dup_mask = vv.duplicated('md5', keep=False)
n_groups = vv[dup_mask]['md5'].nunique()
print(f'Q18  Images belonging to duplicate groups: {dup_mask.sum()}  | duplicate groups: {n_groups}  | removable extra copies: {vv.duplicated("md5").sum()}')
cross = vv.groupby('md5')['label'].nunique()
print(f'     Duplicate groups with CONFLICTING labels (same file in Cat and Dog): {(cross>1).sum()}')

**Q18 – Effect of duplicates:** (1) *Data leakage*: if a copy lands in training and another in validation/test, the model is evaluated on images it has memorised, so metrics are inflated. (2) *Bias/over-weighting*: duplicated images count several times in the loss. (3) *Wasted compute* and over-confident calibration. (4) Duplicates with **conflicting labels** inject label noise. Only exact (byte-identical) duplicates are detected here; near-duplicates (resized/re-encoded copies) would need perceptual hashing — listed as a limitation.

In [ ]:
# Q19: unusual dimensions / file sizes (IQR rule) + very small images + extreme aspect ratios
vv = vv.assign(aspect=vv.width/vv.height, area=vv.width*vv.height)
def iqr_bounds(s):
    q1, q3 = s.quantile([.25,.75]); i = q3-q1; return q1-1.5*i, q3+1.5*i
rows = []
for c in ['width','height','file_size','aspect']:
    lo, hi = iqr_bounds(vv[c]); n = ((vv[c]<lo)|(vv[c]>hi)).sum()
    rows.append((c, round(lo,2), round(hi,2), int(n)))
print(pd.DataFrame(rows, columns=['feature','lower_fence','upper_fence','n_outliers']))
print('\nTiny images (min side < 50 px):', int((vv[['width','height']].min(axis=1) < 50).sum()))
print('Extreme aspect ratio (<0.4 or >2.5):', int(((vv.aspect<0.4)|(vv.aspect>2.5)).sum()))
print('Very small files (<2 KB):', int((vv.file_size<2048).sum()))
odd = vv.assign(minside=vv[['width','height']].min(axis=1)).nsmallest(5,'minside')
odd2 = vv.nlargest(5,'area')
fig, ax = plt.subplots(2, 5, figsize=(16, 6.5))
for a, (_, r) in zip(ax[0], odd.iterrows()):
    with Image.open(r.path) as im: a.imshow(im.convert('RGB'))
    a.set_title(f'smallest {int(r.width)}x{int(r.height)}', fontsize=9); a.axis('off')
for a, (_, r) in zip(ax[1], odd2.iterrows()):
    with Image.open(r.path) as im: a.imshow(im.convert('RGB'))
    a.set_title(f'largest {int(r.width)}x{int(r.height)}', fontsize=9); a.axis('off')
show('q19_anomalies.png')

In [ ]:
# Q20: channel modes (grayscale / RGB / RGBA / palette / CMYK ...)
print(vv['mode'].value_counts().rename('images').to_frame(), '\n')
print(vv['channels'].value_counts().rename('images').to_frame())

**Q20 – Why standardise channels?** A neural network or flattened-pixel model expects an input of fixed shape (H, W, 3). Grayscale images have 1 channel, RGBA 4, palette ('P') images store indices not intensities. Mixed channel counts would crash batching or silently produce feature vectors of different lengths and wrong colour semantics. We therefore convert **everything to 3-channel RGB** (`convert('RGB')`) in the shared preprocessing function.

In [ ]:
# Q21: build the cleaned dataset (drop invalid, cross-class duplicates, exact duplicates, tiny images)
df_clean, steps = data_cleaning.build_cleaned_dataset(df, str(CLEAN), min_side=32)
print(steps.to_frame()); print(f'\nQ21  Images remaining after cleaning: {len(df_clean)}')
print(df_clean.label.value_counts().to_frame('images'))
df_clean.to_csv(RESULTS/'metadata_clean.csv', index=False)

# Phase 4 — Image Preprocessing

**Q22. Why can't ML algorithms process images of different sizes?** Models consume a fixed-length feature vector (or fixed-shape tensor). Flattening a 500×375×3 image gives 562,500 values, a 300×200×3 image gives 180,000 — the columns would not align, a Dense layer's weight matrix has a fixed input size, and mini-batches must be stacked into a single rectangular tensor. All images must therefore share one shape.

In [ ]:
# Q23 + Q24: resize all to 128x128 RGB uint8 arrays
t0 = time.time()
paths = df_clean.path.tolist()
X128 = preprocessing.load_dataset_array(paths, IMG_SIZE)
print(f'Q23/24  X128 shape: {X128.shape}, dtype={X128.dtype}  ({time.time()-t0:.0f}s, {X128.nbytes/1e9:.2f} GB)')
print('One image shape:', X128[0].shape, '(height, width, channels)')
fig, ax = plt.subplots(1, 2, figsize=(8,4))
with Image.open(paths[0]) as im: ax[0].imshow(im.convert('RGB')); ax[0].set_title(f'original {im.size}')
ax[1].imshow(X128[0]); ax[1].set_title('resized 128x128'); [a.axis('off') for a in ax]; show('q23_resize.png')

**Q23 – Choice of 128×128:** large enough to keep ears, snouts and fur texture (the discriminative cues) yet small enough that ~25k images fit in RAM (≈1.2 GB as uint8) and train fast on a Colab GPU. 224×224 would be ~3× more memory/compute for modest gain on this easy task. For traditional ML with flattened pixels we additionally reduce to **64×64** (12,288 features) to keep SVM/KNN/RF tractable (project rule 9.3). Aspect ratio is not preserved (simple squash); letter-boxing/cropping is a possible refinement.

In [ ]:
# Q25: normalisation 0-255 -> 0-1
sample = X128[:200]
sample_n = preprocessing.normalize(sample)
print(f'before: dtype={sample.dtype}  min={sample.min()}  max={sample.max()}  mean={sample.mean():.1f}')
print(f'after : dtype={sample_n.dtype}  min={sample_n.min():.2f}  max={sample_n.max():.2f}  mean={sample_n.mean():.3f}')
print('Traditional ML: normalised explicitly with preprocessing.normalize(). CNN/transfer models: normalisation is the first layer of the model (Rescaling), so the deployed app cannot forget it.')

**Q25 – Why normalise?** Raw 0–255 inputs produce large activations/gradients, slow or unstable optimisation and make distance-based (KNN) and margin-based (SVM, Logistic Regression) methods dominated by scale. Scaling to [0, 1] puts all features on a comparable range, keeps weights/learning rates well-conditioned and speeds convergence. (Pretrained MobileNetV2 specifically expects [-1, 1], which is handled inside the transfer model.)

In [ ]:
# Q26: target encoding Cat=0, Dog=1
df_clean['y'] = df_clean.label.map(LABEL_MAP).astype(int)
y_all = df_clean['y'].values
print(df_clean.groupby(['label','y']).size().to_frame('images'))

In [ ]:
# Q27: stratified 70 / 15 / 15 split (done BEFORE any learned transformation)
from sklearn.model_selection import train_test_split
idx = np.arange(len(df_clean))
idx_tr, idx_tmp = train_test_split(idx, test_size=0.30, stratify=y_all, random_state=SEED)
idx_va, idx_te  = train_test_split(idx_tmp, test_size=0.50, stratify=y_all[idx_tmp], random_state=SEED)
df_clean['split'] = 'train'; df_clean.loc[idx_va,'split']='val'; df_clean.loc[idx_te,'split']='test'
df_clean[['path','orig_path','label','y','split']].to_csv(RESULTS/'splits.csv', index=False)   # reproducibility: saved split list
Y = dict(train=y_all[idx_tr], val=y_all[idx_va], test=y_all[idx_te])
IMG = dict(train=X128[idx_tr], val=X128[idx_va], test=X128[idx_te])
print({k: len(v) for k, v in Y.items()})
print(pd.crosstab(df_clean.split, df_clean.label, normalize='index').round(4)*100)

**Q27 – Proportions (70/15/15):** the large training share gives the CNNs enough data; a 15 % validation set is used for early stopping, architecture choice and model selection; a 15 % test set (~3.7k images) is big enough for a tight confidence interval on accuracy (≈ ±1 %) and stays **untouched until the final evaluation**.

**Q28 – Why stratified?** Stratification keeps the Cat:Dog ratio identical in train, validation and test. Without it, random fluctuation could leave one split with a different class mix, making metrics incomparable across splits (and misleading for precision/recall), especially if a dataset is even slightly imbalanced. It also reduces the variance of the evaluation.

In [ ]:
# Q29: demonstrate data leakage and how it is prevented
# (a) duplicate leakage: split the RAW valid list randomly *without* de-duplicating
raw_valid = df[df.valid].reset_index(drop=True)
a, b = train_test_split(np.arange(len(raw_valid)), test_size=0.3, random_state=SEED)
train_hashes = set(raw_valid.loc[a,'md5'])
leaked = raw_valid.loc[b,'md5'].isin(train_hashes)
print(f'(a) Naive split WITHOUT de-duplication: {int(leaked.sum())} of {len(b)} held-out images have an identical copy in the training set -> inflated scores.')
# (b) transform leakage: scaler fitted on ALL data vs train only
from sklearn.preprocessing import StandardScaler
n_demo = min(3000, len(X128))
Xf = preprocessing.normalize(downsample_2x(X128[:n_demo])).reshape(n_demo, -1)[:, :2000]
cut = int(n_demo * 0.7)
tr_part, te_part = Xf[:cut], Xf[cut:]
leak_mean, ok_mean = StandardScaler().fit(Xf).mean_, StandardScaler().fit(tr_part).mean_
print(f'(b) Scaler fitted on all data differs from train-only scaler by {np.abs(leak_mean-ok_mean).mean():.5f} (mean |diff|): test statistics leaked into preprocessing.')
# (c) our pipeline: verify split disjointness by content hash
h = {s: set(df_clean.loc[df_clean.split==s,'md5']) for s in ['train','val','test']}
print('(c) Cleaned splits - shared hashes train&val / train&test / val&test:', len(h['train']&h['val']), len(h['train']&h['test']), len(h['val']&h['test']))

**Prevention used here:** de-duplicate *before* splitting; split *before* fitting any scaler/PCA (all learned transforms are fit on the training set only); augmentation exists only inside the model and is active only for training; the test set is not touched until the final evaluation; split file list saved in `results/splits.csv`.

# Phase 5 — Traditional Machine Learning (raw pixels, 64×64×3)

In [ ]:
# Q30: flatten pixels
X64 = {s: preprocessing.downsample_2x(IMG[s]) for s in ['train','val','test']}
FEATS = {'raw64': {s: feature_engineering.flatten(preprocessing.normalize(X64[s])) for s in X64}}
print('Image shape:', X64['train'][0].shape)
print(f"Q30  Feature dimensionality for a 64x64 RGB image = 64 x 64 x 3 = {64*64*3}")
print('Feature matrices:', {s: FEATS['raw64'][s].shape for s in FEATS['raw64']})

In [ ]:
# ---- experiment registry: every model is trained on train, scored on VAL during development,
#      and scored on TEST once at the end ----
from sklearn.metrics import confusion_matrix, classification_report, roc_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
registry, _cache, val_rows = {}, {}, []

def register(name, model, kind, feat, train_time):
    registry[name] = dict(model=model, kind=kind, feat=feat, train_time=train_time)

def predict_split(name, split):
    key = (name, split)
    if key not in _cache:
        r = registry[name]; X = FEATS[r['feat']][split]
        if r['kind'] == 'sk':
            _cache[key] = train_ml.sk_scores(r['model'], X)
        else:
            s = r['model'].predict(X, batch_size=128, verbose=0).ravel()
            _cache[key] = ((s >= 0.5).astype(int), s)
    return _cache[key]

def evaluate(name, split):
    pred, score = predict_split(name, split)
    return dict(Model=name, **train_ml.compute_metrics(Y[split], pred, score))

def log_val(name):
    row = evaluate(name, 'val'); row['Train_time_s'] = round(registry[name]['train_time'], 1)
    val_rows.append(row); return row

sub_idx = train_ml.stratified_subsample(Y['train'], SVM_N, SEED)   # fixed subset reused by every SVM
def fit_sk(name, model, feat, subset=False):
    Xtr, ytr = FEATS[feat]['train'], Y['train']
    if subset: Xtr, ytr = Xtr[sub_idx], ytr[sub_idx]
    model, t = train_ml.timed_fit(model, Xtr, ytr)
    register(name, model, 'sk', feat, t); return log_val(name)

# Majority-class baseline for the final comparison table
fit_sk('Majority Baseline', DummyClassifier(strategy='most_frequent'), 'raw64'); val_rows.pop()

In [ ]:
# Q31: Logistic Regression
print(fit_sk('Logistic Regression', make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=300)), 'raw64'))

In [ ]:
# Q32: KNN (compare with Logistic Regression)
print(fit_sk('KNN', KNeighborsClassifier(n_neighbors=5, algorithm='brute', n_jobs=-1), 'raw64'))
print(pd.DataFrame(val_rows)[['Model','Accuracy','F1','ROC_AUC','Train_time_s']])

In [ ]:
# Q33: SVM (RBF) on raw pixels - trained on a stratified subset of the training set (see SVM_N)
print(f'SVM training subset: {len(sub_idx)} images')
print(fit_sk('SVM - Raw Pixels', SVC(kernel='rbf', C=1.0, gamma='scale'), 'raw64', subset=True))

In [ ]:
# Q34: Random Forest
print(fit_sk('Random Forest', RandomForestClassifier(n_estimators=RF_TREES, n_jobs=-1, random_state=SEED), 'raw64'))

In [ ]:
# Q35: comparison table (validation set)
ml_names = ['Logistic Regression','KNN','SVM - Raw Pixels','Random Forest']
tbl35 = pd.DataFrame([r for r in val_rows if r['Model'] in ml_names])
tbl35.round(4)

In [ ]:
# Q36: confusion matrices (validation)
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
errs = {}
for a, n in zip(ax, ml_names):
    cm = confusion_matrix(Y['val'], predict_split(n, 'val')[0])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=a, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    a.set_title(n); a.set_xlabel('Predicted'); a.set_ylabel('True'); errs[n] = int(cm.sum()-np.trace(cm))
show('q36_confusion_ml.png')
print('Misclassifications:', errs); print('Fewest errors ->', min(errs, key=errs.get))

**Q37 – Why raw pixels are hard for traditional ML:** (1) very high dimensionality (12,288 features) relative to samples → curse of dimensionality and over-fitting; (2) pixels are individually meaningless — the signal is in *spatial structure* (edges, shapes, parts), which flat models cannot exploit; (3) no translation/scale/rotation invariance: the same cat shifted by a few pixels yields a completely different vector; (4) huge intra-class variation (pose, lighting, background, breed) vs. small inter-class pixel differences; (5) distance metrics in pixel space (KNN) reflect colour/brightness similarity rather than semantic similarity. CNNs address this via local receptive fields, weight sharing and hierarchical features.

# Phase 6 — Feature Engineering (grayscale, HOG, PCA)

In [ ]:
# Q38: grayscale vs RGB (same models, same splits)
G = {s: feature_engineering.to_grayscale(X64[s]) for s in X64}
FEATS['gray64'] = {s: G[s].reshape(len(G[s]), -1) for s in G}
print('Gray feature dim:', FEATS['gray64']['train'].shape[1], 'vs RGB', FEATS['raw64']['train'].shape[1])
fit_sk('Logistic Regression (Gray)', make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=300)), 'gray64')
fit_sk('KNN (Gray)', KNeighborsClassifier(n_neighbors=5, algorithm='brute', n_jobs=-1), 'gray64')
fit_sk('SVM - Gray Pixels', SVC(kernel='rbf', C=1.0, gamma='scale'), 'gray64', subset=True)
fit_sk('Random Forest (Gray)', RandomForestClassifier(n_estimators=RF_TREES, n_jobs=-1, random_state=SEED), 'gray64')
vt = pd.DataFrame(val_rows).set_index('Model')
cmp38 = pd.DataFrame({'RGB_Acc': [vt.loc[m,'Accuracy'] for m in ml_names],
                      'Gray_Acc': [vt.loc[g,'Accuracy'] for g in ['Logistic Regression (Gray)','KNN (Gray)','SVM - Gray Pixels','Random Forest (Gray)']],
                      'RGB_F1': [vt.loc[m,'F1'] for m in ml_names],
                      'Gray_F1': [vt.loc[g,'F1'] for g in ['Logistic Regression (Gray)','KNN (Gray)','SVM - Gray Pixels','Random Forest (Gray)']]}, index=ml_names)
cmp38['Acc_diff(Gray-RGB)'] = cmp38.Gray_Acc - cmp38.RGB_Acc
print(cmp38.round(4)); print(f"\nMean accuracy: RGB {cmp38.RGB_Acc.mean():.4f} vs Gray {cmp38.Gray_Acc.mean():.4f}. Gray has 3x fewer features and is faster; colour adds little class information for cat-vs-dog (fur colours overlap), so the difference is usually small.")

In [ ]:
# Q39: HOG features + SVM
t0 = time.time()
FEATS['hog'] = {s: feature_engineering.hog_features(G[s]) for s in G}
print(f"HOG feature dim: {FEATS['hog']['train'].shape[1]} (extracted in {time.time()-t0:.0f}s)")
print(fit_sk('HOG + SVM', SVC(kernel='rbf', C=1.0, gamma='scale'), 'hog', subset=True))

In [ ]:
# Q40: HOG+SVM vs Raw+SVM
c40 = pd.DataFrame([r for r in val_rows if r['Model'] in ['SVM - Raw Pixels','HOG + SVM']])
print(c40.round(4))
better = c40.sort_values('Accuracy').iloc[-1]['Model']
print(f'\nBetter approach on validation: {better}')

**Q40 – Interpretation:** HOG describes *local gradient orientations* (edges/contours) in cells with block normalisation, so it captures shape/texture and is partly invariant to illumination and small shifts, while raw pixels encode absolute intensity at fixed positions. HOG features are also far lower-dimensional (1,764 vs 12,288) and less noisy, so the SVM generalises better. Both use the same training subset, so the comparison is controlled. HOG is still a hand-crafted representation, so it remains well below CNNs.

In [ ]:
# Q41: PCA (fit on TRAIN only) - components needed for 90 % / 95 % variance
from sklearn.decomposition import PCA
n_comp = min(PCA_MAX, FEATS['raw64']['train'].shape[1], len(Y['train']))
pca = PCA(n_components=n_comp, svd_solver='randomized', random_state=SEED).fit(FEATS['raw64']['train'])
cum = np.cumsum(pca.explained_variance_ratio_)
def k_for(t): return int(np.argmax(cum >= t) + 1) if cum[-1] >= t else int(n_comp)
k90, k95 = k_for(.90), k_for(.95)
D = FEATS['raw64']['train'].shape[1]
plt.figure(figsize=(7,4)); plt.plot(cum); plt.axhline(.9, ls='--', c='g'); plt.axhline(.95, ls='--', c='r')
plt.xlabel('# components'); plt.ylabel('cumulative explained variance'); plt.title('PCA on raw 64x64x3 pixels'); show('q41_pca.png')
print(f'Original dims: {D}')
print(f'90% variance: {k90} components  -> {100*(1-k90/D):.1f}% reduction')
print(f'95% variance: {k95} components  -> {100*(1-k95/D):.1f}% reduction')
if cum[-1] < .95: print('(note: cumulative variance of computed components is only', round(cum[-1],3), ')')

In [ ]:
# Q42: classifier on PCA features (transform only on val/test)
FEATS['pca95'] = {s: pca.transform(FEATS['raw64'][s])[:, :k95] for s in ['train','val','test']}
print('PCA feature shape:', FEATS['pca95']['train'].shape)
fit_sk('PCA + SVM', SVC(kernel='rbf', C=1.0, gamma='scale'), 'pca95', subset=True)
fit_sk('PCA + Logistic Regression', make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=300)), 'pca95')
print(pd.DataFrame([r for r in val_rows if r['Model'] in ['SVM - Raw Pixels','PCA + SVM','Logistic Regression','PCA + Logistic Regression']]).round(4))

**Q42 – Interpretation:** PCA compresses the 12,288 pixel features into a few hundred orthogonal components with little information loss. Performance stays about the same as on raw pixels (it can slightly improve by removing noise), while training/prediction becomes much faster and memory-light. It does *not* add new discriminative information — PCA is unsupervised and linear — so it cannot close the gap to CNNs.

# Phase 7 — Deep Learning with CNN

In [ ]:
# Q43: baseline CNN (Input -> [Conv2D+ReLU -> MaxPool]x3 -> Flatten -> Dense -> Dropout -> Sigmoid)
FEATS['img128'] = IMG
cfgs = train_cnn.ARCHS; names = list(cfgs)
cnn_A = train_cnn.build_cnn(cfgs[names[0]], name='cnn_A')
cnn_A.summary()

**Q43 – Purpose of each layer**
- **Rescaling (1/255):** normalises pixels to 0–1 inside the model.
- **Conv2D + ReLU:** learns small filters (3×3) that detect local patterns — edges/colours in early layers, textures/parts (ears, eyes, fur) in deeper layers; ReLU adds non-linearity and avoids vanishing gradients.
- **MaxPooling2D (2×2):** halves spatial size, keeps the strongest response → fewer parameters, some translation invariance, larger receptive field.
- **Flatten:** turns the final feature maps into a 1-D vector for the classifier.
- **Dense(128, ReLU):** learns non-linear combinations of the high-level features.
- **Dropout:** randomly zeroes units during training to reduce over-fitting.
- **Dense(1, sigmoid):** outputs P(Dog) ∈ (0,1); trained with binary cross-entropy.

In [ ]:
# Q44: trainable parameters (verified by hand)
trainable = int(sum(np.prod(w.shape) for w in cnn_A.trainable_weights))
print('Trainable parameters (Keras):', f'{trainable:,}   | total:', f'{cnn_A.count_params():,}')
cin, h, hand = 3, 128, 0
for f in cfgs[names[0]]['filters']:
    p = (3*3*cin + 1) * f; hand += p; print(f'Conv {cin}->{f}: (3*3*{cin}+1)*{f} = {p:,}'); cin = f; h //= 2
flat = h*h*cin; d1 = (flat+1)*128; d2 = 129; hand += d1 + d2
print(f'Flatten {h}x{h}x{cin} = {flat:,}\nDense(128): ({flat}+1)*128 = {d1:,}\nOutput: 129\nHand-calculated total = {hand:,}  (matches Keras: {hand==trainable})')

In [ ]:
# Q45: train baseline CNN with EarlyStopping + ModelCheckpoint, plot curves
hist_store, time_store = {}, {}
def plot_hist(h, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h['accuracy'], label='train'); ax[0].plot(h['val_accuracy'], label='val'); ax[0].set_title(title+' - accuracy'); ax[0].legend(); ax[0].set_xlabel('epoch')
    ax[1].plot(h['loss'], label='train'); ax[1].plot(h['val_loss'], label='val'); ax[1].set_title(title+' - loss'); ax[1].legend(); ax[1].set_xlabel('epoch')
    show(f'curves_{title[:12].replace(" ","_")}.png')

hist, t = train_cnn.compile_and_fit(cnn_A, names[0], IMG['train'], Y['train'], IMG['val'], Y['val'], str(MODELS), epochs=EPOCHS_CNN)
hist_store[names[0]] = hist.history; time_store[names[0]] = t
register(names[0], cnn_A, 'keras', 'img128', t); log_val(names[0])
plot_hist(hist.history, 'Baseline CNN')
h = hist.history; gap = h['accuracy'][-1] - h['val_accuracy'][-1]
print(f"Final train acc {h['accuracy'][-1]:.3f} | val acc {h['val_accuracy'][-1]:.3f} | gap {gap:+.3f}")
print('Diagnosis:', 'OVERFITTING (train >> val, val loss rising)' if gap > 0.07 else ('UNDERFITTING (both low)' if h['val_accuracy'][-1] < 0.7 else 'reasonable fit with mild over-fitting'))

**Q45 – Reading the curves:** training accuracy typically keeps rising while validation accuracy plateaus and validation loss turns upward after a few epochs — the classic signature of **over-fitting** (the network memorises training images). Both curves staying low would indicate underfitting. EarlyStopping with `restore_best_weights=True` returns the weights from the best validation epoch; augmentation/dropout (Phase 8) are the remedies.

In [ ]:
# Q46: three architectures (A = baseline above, B = deeper+BN+GAP, C = 5x5 small)
for n in names[1:]:
    m = train_cnn.build_cnn(cfgs[n], name=n.split()[0].replace('-','_'))
    h_, t_ = train_cnn.compile_and_fit(m, n, IMG['train'], Y['train'], IMG['val'], Y['val'], str(MODELS), epochs=EPOCHS_CNN)
    hist_store[n], time_store[n] = h_.history, t_
    register(n, m, 'keras', 'img128', t_); log_val(n)
rows46 = []
for n in names:
    r = evaluate(n, 'val'); r['Params'] = registry[n]['model'].count_params(); r['Train_time_s'] = round(time_store[n],1)
    r['Train_acc(last)'] = hist_store[n]['accuracy'][-1]; rows46.append(r)
t46 = pd.DataFrame(rows46).set_index('Model'); print(t46.round(4))
best_cnn = t46['Accuracy'].idxmax(); print('\nQ46  Best architecture on validation accuracy:', best_cnn)
plt.figure(figsize=(8,4))
for n in names: plt.plot(hist_store[n]['val_accuracy'], label=n)
plt.legend(fontsize=8); plt.title('Validation accuracy per architecture'); plt.xlabel('epoch'); show('q46_arch_compare.png')

**Q46 – Discussion:** compare the table above: more conv layers/filters and BatchNorm + GlobalAveragePooling (CNN-B) usually give better features with far fewer parameters in the dense head (less over-fitting) than Flatten-based heads (CNN-A, CNN-C); larger 5×5 kernels with fewer filters (CNN-C) tend to be cheaper but weaker. The best model is chosen on **validation** accuracy only; the test set is not used for selection.

# Phase 8 — Data Augmentation & Transfer Learning

In [ ]:
# Q47: augmentation (rotation, flip, zoom, shift, shear) - training only
best_cfg = cfgs[best_cnn]
aug = train_cnn.augmentation_layers()
fig, ax = plt.subplots(2, 6, figsize=(16, 5.5))
img0 = (IMG['train'][0:1]/255.).astype('float32')
for a in ax.ravel():
    a.imshow(np.clip(aug(img0, training=True)[0].numpy(), 0, 1)); a.axis('off')
ax[0,0].set_title('augmented variants of one image', loc='left'); show('q47_aug_examples.png')
cnn_aug = train_cnn.build_cnn(best_cfg, augment=True, name='cnn_aug')
h_, t_ = train_cnn.compile_and_fit(cnn_aug, 'CNN + Augmentation', IMG['train'], Y['train'], IMG['val'], Y['val'], str(MODELS), epochs=EPOCHS_AUG, patience=6)
hist_store['CNN + Augmentation'] = h_.history; time_store['CNN + Augmentation'] = t_
register('CNN + Augmentation', cnn_aug, 'keras', 'img128', t_); log_val('CNN + Augmentation')
plot_hist(h_.history, 'CNN with augmentation')
def gap_of(n): h = hist_store[n]; i = int(np.argmin(h['val_loss'])); return h['accuracy'][i]-h['val_accuracy'][i], h['val_loss'][i]
cmp47 = pd.DataFrame({n: {**{k: v for k, v in evaluate(n,'val').items() if k!='Model'},
                          'train-val acc gap (best epoch)': gap_of(n)[0], 'best val_loss': gap_of(n)[1]} for n in [best_cnn, 'CNN + Augmentation']}).T
print(cmp47.round(4))

**Q47 – Before vs after:** augmentation creates new plausible variants each epoch (flips, rotations, zooms, shifts, shears), so the network cannot memorise exact pixels: the **train–val gap shrinks** and validation loss usually drops, at the cost of slower convergence (hence more epochs). Augmentation layers are inactive at validation/test/inference time, so evaluation stays clean.

In [ ]:
# Q48: transfer learning with MobileNetV2 (ImageNet) - frozen head, then fine-tune top layers
tl_model, base = train_cnn.build_transfer_model(PRETRAINED)
h_, t_ = train_cnn.compile_and_fit(tl_model, 'TL frozen', IMG['train'], Y['train'], IMG['val'], Y['val'], str(MODELS), epochs=EPOCHS_TL, lr=1e-3)
hist_store['TL frozen'] = h_.history; time_store['TL frozen'] = t_
frozen_copy = keras.models.load_model(MODELS/'TL_frozen.keras')      # separate copy of the best frozen-head model
register('Transfer Learning (frozen)', frozen_copy, 'keras', 'img128', t_); log_val('Transfer Learning (frozen)')
# fine-tune the top 30 backbone layers with a small learning rate (BatchNorm stays frozen)
train_cnn.unfreeze_top_layers(base, 30)
h2, t2 = train_cnn.compile_and_fit(tl_model, 'TL fine-tuned', IMG['train'], Y['train'], IMG['val'], Y['val'], str(MODELS), epochs=EPOCHS_FT, lr=1e-5, patience=3)
hist_store['TL fine-tuned'] = h2.history; time_store['TL fine-tuned'] = t2
register('Transfer Learning', tl_model, 'keras', 'img128', t_ + t2); log_val('Transfer Learning')
plot_hist(h2.history, 'TL fine-tuning')
print(pd.DataFrame([r for r in val_rows if r['Model'] in [best_cnn, 'CNN + Augmentation', 'Transfer Learning (frozen)', 'Transfer Learning']]).round(4))

**Q48 – Comparison:** the pretrained MobileNetV2 already knows edges, textures, animal parts and even cats/dogs from ImageNet, so a tiny classification head on frozen features beats a CNN trained from scratch on ~17k images, and fine-tuning the top layers with a small learning rate adds a little more. Transfer learning needs fewer epochs and generalises better; the cost is a larger model file and dependency on a pretrained backbone.

## Final model selection (validation only) and the one-time test evaluation
The test set has not been used so far. Deep-learning model selection uses **validation ROC-AUC/accuracy**; then every model in the project is scored **once** on the test set to produce the Section 12 table.

In [ ]:
# model selection among DL models on VALIDATION
dl_names = [best_cnn, 'CNN + Augmentation', 'Transfer Learning']
vsel = pd.DataFrame([evaluate(n,'val') for n in dl_names]).set_index('Model')
final_name = vsel['ROC_AUC'].idxmax()
print(vsel.round(4)); print('\nSelected final model (validation ROC-AUC):', final_name)

In [ ]:
# Section 12 final comparison table on the TEST set (+ time, size, inference speed)
import joblib, tempfile
order = [('Majority / Random Baseline','Majority Baseline'),('Logistic Regression','Logistic Regression'),('KNN','KNN'),
         ('SVM - Raw Pixels','SVM - Raw Pixels'),('Random Forest','Random Forest'),('HOG + SVM','HOG + SVM'),
         ('PCA + Classifier','PCA + SVM'),('Custom CNN', best_cnn),('CNN + Augmentation','CNN + Augmentation'),
         ('Transfer Learning','Transfer Learning'),('Transfer Learning (frozen backbone, extra)','Transfer Learning (frozen)')]
def size_mb(n):
    r = registry[n]
    if r['kind'] == 'keras':
        safe = ''.join(c if c.isalnum() else '_' for c in {'Transfer Learning':'TL fine-tuned','Transfer Learning (frozen)':'TL frozen'}.get(n, n))
        p = MODELS/f'{safe}.keras'; return p.stat().st_size/1e6 if p.exists() else np.nan
    with tempfile.NamedTemporaryFile(suffix='.pkl') as f:
        joblib.dump(r['model'], f.name, compress=3); return os.path.getsize(f.name)/1e6
def infer_ms(n, k=200):
    r = registry[n]; X = FEATS[r['feat']]['test'][:k]; t0 = time.time()
    (r['model'].predict(X) if r['kind']=='sk' else r['model'].predict(X, batch_size=64, verbose=0)); return (time.time()-t0)/len(X)*1000
final_rows = []
for label, n in order:
    row = {'Approach': label, **{k: v for k, v in evaluate(n, 'test').items() if k!='Model'}}
    row['Train_time_s'] = round(registry[n]['train_time'], 1); row['Model_MB'] = round(size_mb(n), 1); row['Inference_ms/img'] = round(infer_ms(n), 3)
    final_rows.append(row)
final_df = pd.DataFrame(final_rows).set_index('Approach')
final_df.to_csv(RESULTS/'final_model_comparison.csv'); final_df.round(4)

In [ ]:
# Q49: detailed evaluation of the DL models on TEST: ROC curves + confusion matrices + classification reports
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
for n in dl_names:
    fpr, tpr, _ = roc_curve(Y['test'], predict_split(n,'test')[1]); ax.plot(fpr, tpr, label=f"{n} (AUC={final_df.loc[{best_cnn:'Custom CNN','CNN + Augmentation':'CNN + Augmentation','Transfer Learning':'Transfer Learning'}[n],'ROC_AUC']:.3f})")
ax.plot([0,1],[0,1],'k--'); ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate'); ax.legend(fontsize=7); ax.set_title('ROC curves (test)'); show('q49_roc.png')
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, n in zip(ax, dl_names):
    sns.heatmap(confusion_matrix(Y['test'], predict_split(n,'test')[0]), annot=True, fmt='d', cmap='Blues', cbar=False, ax=a, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES); a.set_title(n, fontsize=9); a.set_xlabel('Predicted'); a.set_ylabel('True')
show('q49_confusion_dl.png')
print('Classification report -', final_name); print(classification_report(Y['test'], predict_split(final_name,'test')[0], target_names=CLASS_NAMES, digits=4))

In [ ]:
# Q49: sample predictions (selected model, test set)
pred_f, score_f = predict_split(final_name, 'test')
rs = np.random.RandomState(7).choice(len(Y['test']), 16, replace=False)
fig, ax = plt.subplots(2, 8, figsize=(20, 6))
for a, i in zip(ax.ravel(), rs):
    a.imshow(IMG['test'][i]); ok = pred_f[i]==Y['test'][i]
    a.set_title(f"T:{CLASS_NAMES[Y['test'][i]]} P:{CLASS_NAMES[pred_f[i]]}\nP(Dog)={score_f[i]:.2f}", color='green' if ok else 'red', fontsize=9); a.axis('off')
show('q49_sample_predictions.png')

# 13. Error Analysis

In [ ]:
# Which images does the selected model get wrong? (test set)
from scipy.ndimage import laplace
te_df = df_clean.iloc[idx_te].reset_index(drop=True).copy()
te_df['pred'] = pred_f; te_df['p_dog'] = score_f; te_df['correct'] = te_df.pred == te_df.y
gray_te = feature_engineering.to_grayscale(IMG['test'])
te_df['brightness'] = IMG['test'].reshape(len(te_df), -1).mean(1)
te_df['contrast'] = gray_te.reshape(len(te_df), -1).std(1)
te_df['sharpness'] = [laplace(g).var() for g in gray_te]
te_df['aspect'] = te_df.width/te_df.height
te_df['min_side'] = te_df[['width','height']].min(axis=1)
te_df['confidence'] = np.maximum(te_df.p_dog, 1-te_df.p_dog)
wrong = te_df[~te_df.correct]
print(f'Test errors: {len(wrong)} / {len(te_df)} ({len(wrong)/len(te_df)*100:.2f}%)')
print('Cat->Dog errors:', int(((wrong.y==0)&(wrong.pred==1)).sum()), '| Dog->Cat errors:', int(((wrong.y==1)&(wrong.pred==0)).sum()))
print('\nMean image properties: correct vs misclassified')
print(te_df.groupby('correct')[['brightness','contrast','sharpness','aspect','min_side','confidence']].mean().rename(index={True:'correct', False:'wrong'}).round(3))
print('\nError rate by brightness quartile:'); print(te_df.groupby(pd.qcut(te_df.brightness, 4))['correct'].apply(lambda s: 1-s.mean()).round(4))
print('\nError rate by sharpness quartile:'); print(te_df.groupby(pd.qcut(te_df.sharpness, 4, duplicates='drop'))['correct'].apply(lambda s: 1-s.mean()).round(4))

In [ ]:
# the most confident mistakes (hardest cases)
hard = wrong.sort_values('confidence', ascending=False).head(12)
fig, ax = plt.subplots(2, 6, figsize=(18, 6.5))
for a, (_, r) in zip(ax.ravel(), hard.iterrows()):
    with Image.open(r.path) as im: a.imshow(im.convert('RGB'))
    a.set_title(f"True {CLASS_NAMES[r.y]} -> Pred {CLASS_NAMES[r.pred]}\nconf {r.confidence:.2f}", fontsize=9, color='red'); a.axis('off')
show('q49_hard_cases.png')

**Error-analysis interpretation (check against the images/statistics printed above):** typical failure groups are (1) images where the animal is small, partially cut off or occluded (e.g. only a paw/ear visible); (2) multiple animals or people in the frame, or a cat and dog together; (3) unusual poses and extreme close-ups; (4) very dark/over-exposed, blurry or low-resolution pictures (compare the brightness/sharpness quartile error rates); (5) breed look-alikes (small fluffy dogs vs. cats, large cats vs. small dogs); (6) non-photographic content or mislabeled images in the dataset (a "confident mistake" is often a label error). Remedies: stronger augmentation, higher input resolution, ensembling, cleaning noisy labels, more diverse data, and Grad-CAM to check what the model attends to.

# Phase 9 — Final Deployment (Q50)

In [ ]:
# Save the selected model + baselines
final_key = {best_cnn: best_cnn, 'CNN + Augmentation': 'CNN + Augmentation', 'Transfer Learning': 'Transfer Learning'}[final_name]
registry[final_key]['model'].save(MODELS/'final_model.keras')
print(f'Saved final model ({final_name}): {(MODELS/"final_model.keras").stat().st_size/1e6:.1f} MB')
for n in ['Logistic Regression','SVM - Raw Pixels','HOG + SVM','PCA + SVM']:       # KNN / RF skipped: KNN stores the whole training set, RF is very large
    joblib.dump(registry[n]['model'], MODELS/(n.replace(' ','_').replace('+','plus')+'.pkl'), compress=3)
joblib.dump(pca, MODELS/'pca_raw64.pkl', compress=3)
json.dump({'final_model': final_name, 'img_size': IMG_SIZE, 'classes': CLASS_NAMES, 'preprocessing': 'RGB, bilinear resize 128x128, normalization inside model (Rescaling)', 'seed': SEED},
          open(MODELS/'model_card.json','w'), indent=2)
print(sorted(p.name for p in MODELS.iterdir()))

In [ ]:
# reload WITHOUT retraining + predict on three unseen test images (via the same function the app uses)
importlib.reload(predict)
loaded = predict.load_trained_model(MODELS/'final_model.keras')
unseen = te_df.sample(3, random_state=3)
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, (_, r) in zip(ax, unseen.iterrows()):
    res = predict.predict_image(loaded, r.path)
    a.imshow(res['image']); a.axis('off')
    a.set_title(f"True: {CLASS_NAMES[r.y]}\nPred: {res['label']}  (P(Dog)={res['prob_dog']:.3f}, conf={res['confidence']*100:.1f}%)", fontsize=9)
show('q50_unseen_predictions.png')
# invalid upload must be handled gracefully
bad_file = Path('/tmp/not_an_image.jpg'); bad_file.write_text('this is not an image')
try: predict.predict_image(loaded, bad_file)
except preprocessing.InvalidImageError as e: print('Invalid upload handled gracefully ->', e)

In [ ]:
# README (execution steps) - app/app.py, requirements.txt were written with the project files
readme = '''# Cat vs Dog Image Classification

End-to-end project: EDA -> cleaning -> preprocessing -> ML baselines -> HOG/PCA -> CNN -> augmentation -> transfer learning -> Streamlit app.

## Reproduce
1. `pip install -r requirements.txt` (Python 3.10+; GPU recommended for training)
2. Open `notebooks/Cat_vs_Dog_Capstone.ipynb` in Google Colab, mount Drive, set `DATASET_DIR`, run all cells.
   Outputs: `models/`, `results/` (metadata, splits.csv, final_model_comparison.csv), `figures/`.
3. Seed is fixed (42); the split file list is saved in `results/splits.csv`.

## Run the app locally
```
cd Cat-Dog-Classification
streamlit run app/app.py
```
Upload a JPG/PNG: the app shows the image, predicted class, P(Cat)/P(Dog) and confidence. Invalid files are rejected with a message.
The app loads `models/final_model.keras` (no retraining) and uses `src/preprocessing.py`, the same preprocessing used for training.

## Structure
`src/` reusable modules - `app/app.py` Streamlit app - `models/` saved models - `results/` tables - `figures/` plots
'''
(PROJECT_DIR/'README.md').write_text(readme)
if not (PROJECT_DIR/'requirements.txt').exists():
    (PROJECT_DIR/'requirements.txt').write_text('numpy\npandas\nmatplotlib\nseaborn\npillow\nscikit-learn\nscikit-image\nscipy\njoblib\ntensorflow>=2.16\nkeras>=3\nstreamlit\n')
print('README written')

In [ ]:
# write app/app.py and requirements into the project (copy of the files shipped with this notebook)
APP_CODE = r'''
import sys
from pathlib import Path
import pandas as pd
import streamlit as st
from PIL import Image

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from predict import load_trained_model, predict_image
from preprocessing import InvalidImageError

MODEL_PATH = ROOT / "models" / "final_model.keras"

st.set_page_config(page_title="Cat vs Dog Classifier", page_icon="🐾")
st.title("🐱 Cat vs Dog 🐶 Image Classifier")
st.caption("Upload an image; the saved model predicts Cat or Dog (no retraining).")


@st.cache_resource
def get_model():
    return load_trained_model(MODEL_PATH)


try:
    model = get_model()
except Exception as e:
    st.error(f"Could not load model at {MODEL_PATH}: {e}")
    st.stop()

min_conf = st.sidebar.slider("Confidence needed (below = 'Uncertain')", 0.50, 0.95, 0.60, 0.01)
if "history" not in st.session_state:
    st.session_state.history = []

up = st.file_uploader("Upload a Cat or Dog image", type=["jpg", "jpeg", "png", "bmp", "webp"])
if up is not None:
    try:
        res = predict_image(model, up)
    except InvalidImageError as e:
        st.error(f"Invalid image file. Please upload a valid JPG/PNG. ({e})")
    except Exception as e:
        st.error(f"Unexpected error while predicting: {e}")
    else:
        up.seek(0)
        st.image(Image.open(up), caption="Uploaded image", use_container_width=True)
        label = res["label"] if res["confidence"] >= min_conf else "Uncertain"
        st.subheader(f"Prediction: {label}")
        c1, c2, c3 = st.columns(3)
        c1.metric("P(Dog)", f"{res['prob_dog']:.3f}")
        c2.metric("P(Cat)", f"{res['prob_cat']:.3f}")
        c3.metric("Confidence", f"{res['confidence']*100:.1f}%")
        st.progress(float(res["prob_dog"]), text="Probability of Dog")
        st.session_state.history.append(dict(file=up.name, prediction=label,
                                             prob_dog=round(res["prob_dog"], 4),
                                             confidence=round(res["confidence"], 4)))

if st.session_state.history:
    st.markdown("### Prediction history (this session)")
    st.dataframe(pd.DataFrame(st.session_state.history))

'''
(PROJECT_DIR/'app'/'app.py').write_text(APP_CODE.strip('\n') + '\n')
print('app/app.py written')

In [ ]:
# Launch the Streamlit app from Colab (public tunnel). Open the printed URL; the tunnel password is your Colab VM's public IP.
if IN_COLAB:
    !pip -q install streamlit
    !npm install -g localtunnel > /dev/null 2>&1
    import subprocess, urllib.request
    print('Tunnel password (public IP):', urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip())
    subprocess.Popen(['streamlit','run',str(PROJECT_DIR/'app'/'app.py'),'--server.port','8501','--server.headless','true'])
    time.sleep(8)
    !npx localtunnel --port 8501
else:
    print('Local run:  streamlit run', PROJECT_DIR/'app'/'app.py')

# Summary & deliverables
Everything is saved under `Cat-Dog-Classification/` in your Drive: `src/`, `app/app.py`, `models/`, `results/final_model_comparison.csv` (+ cleaned metadata, splits, corrupted-image log), `figures/`, `README.md`, `requirements.txt`.
Use the printed tables, curves and error analysis for your report (PDF/DOCX) and 5–10 minute presentation.